In [93]:
import os
import json
import polars as pl

In [95]:
data_raw = "data_raw/segments"

In [97]:
os.makedirs(data_raw, exist_ok=True)

In [99]:
done = [x.replace('.json','') for x in os.listdir(data_raw)]

In [101]:
source_folders = os.listdir('downloads/segments')

In [103]:
source_folders

['2024-12-24',
 '2024-12-25',
 '2024-12-26',
 '2024-12-27',
 '2024-12-28',
 '2024-12-29',
 '2024-12-30',
 '2024-12-31',
 '2025-01-01',
 '2025-01-02',
 '2025-01-03',
 '2025-01-04',
 '2025-01-05',
 '2025-01-06',
 '2025-01-07']

In [105]:
for s in source_folders:
    if s not in done:
        print(s)
        day = []
        for f in os.listdir(f'downloads/segments/{s}'):
            time = f.split('_')[1].split('.')[0]
            with open(os.path.join(f'downloads/segments/{s}', f), 'r', encoding='utf-8') as file:
                segment = json.loads(file.read())
                segment['date'] = time
                day.append(segment)
        df = pl.DataFrame(day)
        df.write_parquet(os.path.join(data_raw, f'{s}.parquet'))
        with open(os.path.join(data_raw, f'{s}.json'), 'w+', encoding='utf-8') as export:
            export.write(json.dumps(day))

In [155]:
overview = df.group_by(["name","activity_type","country","city","distance","total_elevation_gain"]).agg(pl.col("effort_count").max()).sort("effort_count", descending=True)
overview

name,activity_type,country,city,distance,total_elevation_gain,effort_count
str,str,str,str,f64,f64,i64
"""Richmond ITT1""","""Ride""","""United Kingdom""","""London""",589.3,7.2,4625714
"""Box Hill 2.2k""","""Ride""","""United Kingdom""","""Mole Valley District, Surrey, …",2296.62,119.1,1415736
"""Die Eine Runde""","""Ride""","""Czech Republic""","""Brno""",380.0,0.0,1089698
"""Overtake the Boris Bikes""","""Run""","""United Kingdom""","""London""",591.8,2.2,663772
"""Třebešín-dráha K""","""Ride""","""Czech Republic""","""Prague""",347.2,0.0,402245
"""Alpe d'Huez""","""Ride""","""France""","""Le Bourg D'Oisans""",12024.9,1047.3,336375
"""Sa Calobra - Coll dels Reis""","""Ride""","""Spain""","""Escorca""",9416.77,659.8,323413
"""Opičí Časovka, Podolská vodárn…","""Ride""","""Czech Republic""","""Prague""",1390.1,6.0,323017
"""Prostějov kolo dráha P""","""Ride""","""Czech Republic""","""Prostějov""",311.1,0.0,251263


In [157]:
pl.Config(tbl_rows=300)

In [187]:
last_date = df.select(pl.col('date')).max().item()
last_date = last_date.strftime("%Y-%m-%d")
last_date

'2025-01-07'

In [159]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    overview_markdown = repr(overview)

In [161]:
overview_markdown

"shape: (261, 7)\n| name              | activity_type | country        | city             | distance | total_elevation_ | effort_count |\n| ---               | ---           | ---            | ---              | ---      | gain             | ---          |\n| str               | str           | str            | str              | f64      | ---              | i64          |\n|                   |               |                |                  |          | f64              |              |\n|-------------------|---------------|----------------|------------------|----------|------------------|--------------|\n| Richmond ITT1     | Ride          | United Kingdom | London           | 589.3    | 7.2              | 4625714      |\n| Box Hill 2.2k     | Ride          | United Kingdom | Mole Valley      | 2296.62  | 119.1            | 1415736      |\n|                   |               |                | District,        |          |                  |              |\n|                   | 

In [189]:
with open('README.md', "w+", encoding="utf8") as readme:
    content = readme.read()
    content = content.split("## Poslední aktualizace")[0] 
    readme.write(f"""{content}\n\n## Poslední aktualizace\n\n{last_date}\n\n##Sledované segmenty\n\n" {overview_markdown}""")

In [193]:
df = pl.read_parquet('data_raw/segments/*.parquet')

In [83]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth.
    Returns distance in kilometers.
    """
    R = 6371  # Earth's radius in kilometers

    # Convert latitude and longitude to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Differences in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine formula
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

calculate_distance = lambda row: haversine_distance(
    row['souradnice_zacatek'][1],  # latitude1
    row['souradnice_zacatek'][0],  # longitude1
    row['souradnice_konec'][1],    # latitude2
    row['souradnice_konec'][0]     # longitude2
)

In [203]:
df = df.with_columns(pl.struct(['start_latlng','end_latlng']) \
       .map_elements(lambda x: haversine_distance(x['start_latlng'][0], x['start_latlng'][1], x['end_latlng'][0], x['end_latlng'][1])).alias('start_to_finish_distance'))

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredict

In [209]:
df.group_by('name').agg(pl.col('start_to_finish_distance').max()).sort('start_to_finish_distance')

name,start_to_finish_distance
str,f64
"""Prostějov kolo dráha P""",0.000451
"""400 metrů, 44 120 diváků""",0.00052
"""Pražačka 320""",0.000925
"""400m-Jičín""",0.001068
"""Track Domažlice""",0.001114
"""Benedikt - Parkoviště""",0.001136
"""Lužánky - Plochá dráha""",0.001678
"""Tatran Třemošná""",0.002124
"""Die Eine Runde""",0.002446


In [195]:
df.with_columns(
   pl.struct(["start_latlng", "end_latlng"]).map(lambda x: haversine_distance(x["start_latlng"][0], x["start_latlng"][1], x["end_latlng"][0], x["end_latlng"][1])).alias("start_to_finish")
)

AttributeError: 'Expr' object has no attribute 'map'

In [91]:
df.with_columns(
    pl.map_elements(lambda x, y: calculate_distance(x, y), pl.col("start_latlng"), pl.col("end_latlng")).alias("start_to_finish")
)

AttributeError: module 'polars' has no attribute 'map_elements'

In [ ]:
df.with_columns(pl.struct(['start_latlng','end_latlng']) \
       .map_elements(lambda x: func(x['col_1'], x['col_2'])).alias('col_3'))

In [51]:
import datetime

In [52]:
df

id,resource_state,name,activity_type,distance,average_grade,maximum_grade,elevation_high,elevation_low,start_latlng,end_latlng,elevation_profile,elevation_profiles,climb_category,city,state,country,private,hazardous,starred,created_at,updated_at,total_elevation_gain,map,effort_count,athlete_count,star_count,athlete_segment_stats,xoms,local_legend,date
i64,i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,struct[2],i64,str,str,str,bool,bool,bool,str,str,f64,struct[3],i64,i64,i64,struct[6],struct[4],struct[7],str
10186045,3,"""Opičí Časovka, Podolská vodárn…","""Ride""",1390.1,-0.1,4.6,194.6,189.3,"[50.059206, 14.419278]","[50.047279, 14.413948]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/56I7UPBBYBG67HREOLVKL5SAZ7JTJTIKO2UBRDD2BUOQJMSVE7NUOIBMNQOAT272OPNXXURJX6NUXR2ALNXJRY3V"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/VSC3NTJUWQX2WJVBYRDYI3S4VGTY4MVCEA5FDT7XOGSV6QFTTZKVCPX753LZ4IZLORUQCG76YIJXNB3PGVVSTZH5""}",0,"""Prague""","""Hlavní město Praha""","""Czech Republic""",false,false,false,"""2015-08-10T23:24:43Z""","""2021-05-20T08:03:30Z""",6.0,"{""s10186045"",""_eppHmg_wAbB@r@CbAB~@NnAV^BdC`@`Dv@~Al@hDtAdAh@lClAJJzAn@t@b@xE|BNDz@\bBz@fEhBdAf@PDfAh@vBz@h@X^X|@Xn@Z"",3}",322708,16496,57,"{null,null,null,null,null,0}","{""1:33"",""2:06"",""1:33"",{""strava://segments/10186045/leaderboard"",""overall"",""All-Time""}}","{27344262,""Petr Duřt"",""https://graph.facebook.com/2254515128130422/picture?height=256&width=256"",""59 efforts in the last 90 days"",""59"",{""59 efforts"",""42 efforts""},""strava://segments/10186045/local_legend?categories%5B%5D=overall""}","""1735077907"""
10356069,3,"""Kvilda Bučina""","""Ride""",5645.6,2.3,7.0,1165.4,1028.8,"[49.014979, 13.579256]","[48.971456, 13.596955]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/G2CPIBKBE4KIJHQQDT7BL2ZOF6YTBUXTSHDMT6QYMQYOVBXTGHKNT5RQ24OZEEDRSYIN65PVTADGIQZ67VFX7T6X"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/WFBOBN6HVGDWLP3O72XZMR4E76TB4UM3IX62EJ7ESTP6JNAX7GZ3E3SGQIXBP42DXJ7PURHBLOEXI7T3JFYCPI3P""}",0,"""Kvilda""","""South Bohemian Region""","""Czech Republic""",false,false,false,"""2015-08-30T18:42:10Z""","""2021-05-17T08:05:54Z""",139.2,"{""s10356069"",""qfdjHie{qAf@H^Jz@b@hAp@~D~AxAfAxA^h@GxAm@pAeAdAoARgA?kBz@aCnMwQnBwCtAaC|B_Fj@wA^i@b@a@n@WvAErBF|GDVBhB@j@HfFPjCBrBHhB~@xAdCxB~CxAlB|@l@fCf@vBf@xBr@pBf@jAPdFjAdBf@vAh@f@NzAh@tCz@jALrBc@rAe@ZQjJiGl@m@lByBtG{HdBcBr@_@xBWjCh@xBZzBIpB[pBg@v@s@FKfAqCfAyC|@oAb@WrBs@TU\i@pAsCrDkHbG{KxAsBrGgGhGuFh@Qx@AhED"",3}",5604,1683,21,"{null,null,null,null,null,0}","{""10:47"",""13:11"",""10:47"",{""strava://segments/10356069/leaderboard"",""overall"",""All-Time""}}","{3478016,""Tyrel Fuchs"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/3478016/13532402/10/large.jpg"",""2 efforts in the last 90 days"",""2"",{""2 efforts"",""1 effort""},""strava://segments/10356069/local_legend?categories%5B%5D=overall""}","""1735077732"""
10356069,3,"""Kvilda Bučina""","""Ride""",5645.6,2.3,7.0,1165.4,1028.8,"[49.014979, 13.579256]","[48.971456, 13.596955]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/G2CPIBKBE4KIJHQQDT7BL2ZOF6YTBUXTSHDMT6QYMQYOVBXTGHKNT5RQ24OZEEDRSYIN65PVTADGIQZ67VFX7T6X"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/WFBOBN6HVGDWLP3O72XZMR4E76TB4UM3IX62EJ7ESTP6JNAX7GZ3E3SGQIXBP42DXJ7PURHBLOEXI7T3JFYCPI3P""}",0,"""Kvilda""","""South Bohemian Region""","""Czech Republic""",false,false,false,"""2015-08-30T18:42:10Z""","""2021-05-17T08:05:54Z""",139.2,"{""s10356069"",""qfdjHie{qAf@H^Jz@b@hAp@~D~AxAfAxA^h@GxAm@pAeAdAoARgA?kBz@aCnMwQnBwCtAaC|B_Fj@wA^i@b@a@n@WvAErBF|GDVBhB@j@HfFPjCBrBHhB~@xAdCxB~CxAlB|@l@fCf@vBf@xBr@pBf@jAPdFjAdBf@vAh@f@NzAh@tCz@jALrBc@rAe@ZQjJiGl@m@lByBtG{HdBcBr@_@xBWjCh@xBZzBIpB[pBg@v@s@FKfAqCfAyC|@oAb@WrBs@TU\i@pAsCrDkHbG{KxAsBrGgGhGuFh@Qx@AhED"",3}",5604,1683,21,"{null,null,null,null,null,0}","{""10:47"",""13:11"",""10:47"",{""strava://segments/10356069/le

In [53]:
df.with_columns(
    (pl.col("date").cast(pl.Int64))
)

id,resource_state,name,activity_type,distance,average_grade,maximum_grade,elevation_high,elevation_low,start_latlng,end_latlng,elevation_profile,elevation_profiles,climb_category,city,state,country,private,hazardous,starred,created_at,updated_at,total_elevation_gain,map,effort_count,athlete_count,star_count,athlete_segment_stats,xoms,local_legend,date
i64,i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,struct[2],i64,str,str,str,bool,bool,bool,str,str,f64,struct[3],i64,i64,i64,struct[6],struct[4],struct[7],i64
10186045,3,"""Opičí Časovka, Podolská vodárn…","""Ride""",1390.1,-0.1,4.6,194.6,189.3,"[50.059206, 14.419278]","[50.047279, 14.413948]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/56I7UPBBYBG67HREOLVKL5SAZ7JTJTIKO2UBRDD2BUOQJMSVE7NUOIBMNQOAT272OPNXXURJX6NUXR2ALNXJRY3V"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/VSC3NTJUWQX2WJVBYRDYI3S4VGTY4MVCEA5FDT7XOGSV6QFTTZKVCPX753LZ4IZLORUQCG76YIJXNB3PGVVSTZH5""}",0,"""Prague""","""Hlavní město Praha""","""Czech Republic""",false,false,false,"""2015-08-10T23:24:43Z""","""2021-05-20T08:03:30Z""",6.0,"{""s10186045"",""_eppHmg_wAbB@r@CbAB~@NnAV^BdC`@`Dv@~Al@hDtAdAh@lClAJJzAn@t@b@xE|BNDz@\bBz@fEhBdAf@PDfAh@vBz@h@X^X|@Xn@Z"",3}",322708,16496,57,"{null,null,null,null,null,0}","{""1:33"",""2:06"",""1:33"",{""strava://segments/10186045/leaderboard"",""overall"",""All-Time""}}","{27344262,""Petr Duřt"",""https://graph.facebook.com/2254515128130422/picture?height=256&width=256"",""59 efforts in the last 90 days"",""59"",{""59 efforts"",""42 efforts""},""strava://segments/10186045/local_legend?categories%5B%5D=overall""}",1735077907
10356069,3,"""Kvilda Bučina""","""Ride""",5645.6,2.3,7.0,1165.4,1028.8,"[49.014979, 13.579256]","[48.971456, 13.596955]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/G2CPIBKBE4KIJHQQDT7BL2ZOF6YTBUXTSHDMT6QYMQYOVBXTGHKNT5RQ24OZEEDRSYIN65PVTADGIQZ67VFX7T6X"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/WFBOBN6HVGDWLP3O72XZMR4E76TB4UM3IX62EJ7ESTP6JNAX7GZ3E3SGQIXBP42DXJ7PURHBLOEXI7T3JFYCPI3P""}",0,"""Kvilda""","""South Bohemian Region""","""Czech Republic""",false,false,false,"""2015-08-30T18:42:10Z""","""2021-05-17T08:05:54Z""",139.2,"{""s10356069"",""qfdjHie{qAf@H^Jz@b@hAp@~D~AxAfAxA^h@GxAm@pAeAdAoARgA?kBz@aCnMwQnBwCtAaC|B_Fj@wA^i@b@a@n@WvAErBF|GDVBhB@j@HfFPjCBrBHhB~@xAdCxB~CxAlB|@l@fCf@vBf@xBr@pBf@jAPdFjAdBf@vAh@f@NzAh@tCz@jALrBc@rAe@ZQjJiGl@m@lByBtG{HdBcBr@_@xBWjCh@xBZzBIpB[pBg@v@s@FKfAqCfAyC|@oAb@WrBs@TU\i@pAsCrDkHbG{KxAsBrGgGhGuFh@Qx@AhED"",3}",5604,1683,21,"{null,null,null,null,null,0}","{""10:47"",""13:11"",""10:47"",{""strava://segments/10356069/leaderboard"",""overall"",""All-Time""}}","{3478016,""Tyrel Fuchs"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/3478016/13532402/10/large.jpg"",""2 efforts in the last 90 days"",""2"",{""2 efforts"",""1 effort""},""strava://segments/10356069/local_legend?categories%5B%5D=overall""}",1735077732
10356069,3,"""Kvilda Bučina""","""Ride""",5645.6,2.3,7.0,1165.4,1028.8,"[49.014979, 13.579256]","[48.971456, 13.596955]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/G2CPIBKBE4KIJHQQDT7BL2ZOF6YTBUXTSHDMT6QYMQYOVBXTGHKNT5RQ24OZEEDRSYIN65PVTADGIQZ67VFX7T6X"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/WFBOBN6HVGDWLP3O72XZMR4E76TB4UM3IX62EJ7ESTP6JNAX7GZ3E3SGQIXBP42DXJ7PURHBLOEXI7T3JFYCPI3P""}",0,"""Kvilda""","""South Bohemian Region""","""Czech Republic""",false,false,false,"""2015-08-30T18:42:10Z""","""2021-05-17T08:05:54Z""",139.2,"{""s10356069"",""qfdjHie{qAf@H^Jz@b@hAp@~D~AxAfAxA^h@GxAm@pAeAdAoARgA?kBz@aCnMwQnBwCtAaC|B_Fj@wA^i@b@a@n@WvAErBF|GDVBhB@j@HfFPjCBrBHhB~@xAdCxB~CxAlB|@l@fCf@vBf@xBr@pBf@jAPdFjAdBf@vAh@f@NzAh@tCz@jALrBc@rAe@ZQjJiGl@m@lByBtG{HdBcBr@_@xBWjCh@xBZzBIpB[pBg@v@s@FKfAqCfAyC|@oAb@WrBs@TU\i@pAsCrDkHbG{KxAsBrGgGhGuFh@Qx@AhED"",3}",5604,1683,21,"{null,null,null,null,null,0}","{""10:47"",""13:11"",""10:47"",{""strava://segments/10356069/leaderboard"",

In [54]:
df = df.with_columns(
    (pl.col("date").cast(pl.Int64) * 1000000).cast(pl.Datetime(time_zone='CET'))
)

In [55]:
df.sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,1902247
2024-12-25 00:00:00 CET,7384499
2024-12-26 00:00:00 CET,10513226
2024-12-27 00:00:00 CET,18965770
2024-12-28 00:00:00 CET,29467674
…,…
2025-01-03 00:00:00 CET,29497016
2025-01-04 00:00:00 CET,29502458
2025-01-05 00:00:00 CET,29507482


In [56]:
df.filter(pl.col('date').dt.hour() == 23).sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,1902247
2024-12-25 00:00:00 CET,5124263
2024-12-26 00:00:00 CET,3811608
2024-12-27 00:00:00 CET,14698890
2024-12-28 00:00:00 CET,14734637
…,…
2025-01-02 00:00:00 CET,14747390
2025-01-03 00:00:00 CET,14749551
2025-01-04 00:00:00 CET,14752780


In [57]:
df.filter(pl.col("city") == "Brno").filter(pl.col('total_elevation_gain') > 5).filter(pl.col('date').dt.hour() == 23).sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,220886
2024-12-25 00:00:00 CET,443249
2024-12-26 00:00:00 CET,221568
2024-12-27 00:00:00 CET,221618
2024-12-28 00:00:00 CET,221597
…,…
2025-01-02 00:00:00 CET,221819
2025-01-03 00:00:00 CET,221836
2025-01-04 00:00:00 CET,221901


In [58]:
df.filter(pl.col("city") == "Brno").filter(pl.col('date').dt.hour() == 23).sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,269552
2024-12-25 00:00:00 CET,491939
2024-12-26 00:00:00 CET,411880
2024-12-27 00:00:00 CET,1500485
2024-12-28 00:00:00 CET,1500683
…,…
2025-01-02 00:00:00 CET,1501565
2025-01-03 00:00:00 CET,1501616
2025-01-04 00:00:00 CET,1501873


In [59]:
df.filter(pl.col("name") == "Risova studanka - od zavory").filter(pl.col('date').dt.hour() == 23).sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,41326
2024-12-25 00:00:00 CET,82656
2024-12-26 00:00:00 CET,41336
2024-12-27 00:00:00 CET,41351
2024-12-28 00:00:00 CET,41360
…,…
2025-01-02 00:00:00 CET,41388
2025-01-03 00:00:00 CET,41389
2025-01-04 00:00:00 CET,41392


In [60]:
df.filter(pl.col("name") == "tenis 1K").filter(pl.col('date').dt.hour() == 23).sort("date").group_by_dynamic("date", every="1d").agg(pl.col("effort_count").sum())

date,effort_count
"datetime[μs, CET]",i64
2024-12-24 00:00:00 CET,48666
2024-12-25 00:00:00 CET,48690
2024-12-26 00:00:00 CET,48725
2024-12-27 00:00:00 CET,48738
2024-12-28 00:00:00 CET,48761
…,…
2025-01-02 00:00:00 CET,48898
2025-01-03 00:00:00 CET,48917
2025-01-04 00:00:00 CET,48945


In [61]:
df.filter(pl.col("city") == "Brno")

id,resource_state,name,activity_type,distance,average_grade,maximum_grade,elevation_high,elevation_low,start_latlng,end_latlng,elevation_profile,elevation_profiles,climb_category,city,state,country,private,hazardous,starred,created_at,updated_at,total_elevation_gain,map,effort_count,athlete_count,star_count,athlete_segment_stats,xoms,local_legend,date
i64,i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,struct[2],i64,str,str,str,bool,bool,bool,str,str,f64,struct[3],i64,i64,i64,struct[6],struct[4],struct[7],"datetime[μs, CET]"
2645185,3,"""Junglepark""","""Run""",367.37,2.4,24.9,265.6,242.3,"[49.182161, 16.574647]","[49.181618, 16.578665]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/FHRGIFNIZ42D74JPYTZARSW5JHL7MYHIMXLR2AH7XDDWSGWSLK55WGXIMOV656PLSZUKPXJ42XGFNWUVLQBM5GFN"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/ICICBB574XSDR5VKHQNJR3ZM2RT3AA7IMNSEFQAMK3BJP3GBBR3E57ILYUWPGSNUROC7FTYXMRC3WOLTA6WFNBZS""}",0,"""Brno""","""South Moravian Region""","""Czech Republic""",false,false,false,"""2012-10-27T15:21:08Z""","""2021-05-20T08:02:21Z""",25.6568,"{""s2645185"",""o{dkHofddBTs@RgAXuD?e@Da@R{AV_@x@M\i@Bg@Co@Yg@USIMUg@Ok@"",3}",37923,3375,26,"{98,""2023-03-27"",""everyone"",8786467799,""everyone"",23}","{""1:03"",""1:16"",""1:03"",{""strava://segments/2645185/leaderboard"",""overall"",""All-Time""}}","{134435067,""Peter Feckanic"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/134435067/30610914/3/large.jpg"",""37 efforts in the last 90 days"",""37"",{""37 efforts"",""17 efforts""},""strava://segments/2645185/local_legend?categories%5B%5D=overall""}",2024-12-24 23:05:13 CET
7116683,3,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,18.7,400.8,279.9,"[49.228134, 16.500734]","[49.233992, 16.467391]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/UXVGSZHT4EJIYPLYRB5GSH72NXWQ6T5F4PJHUHCI6VJYUJCXBS7OIL2VH3BWZEXEWO5TZCCSUUBLMNIZKQYH6KQ="",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/I7WBMPFRT26U6AEHTVU2J6Z7LXWSIKOVATSNEHLDSYFLXGJ7VRHZHFNIXEMX6BQIZ6EHQSSVTNZJP5PE6L2G54A=""}",1,"""Brno""","""South Moravian Region""","""Czech Republic""",false,false,true,"""2014-04-30T14:24:05Z""","""2021-05-20T08:02:57Z""",138.5,"{""s7116683"",""yzmkHqxucBKdAMn@Cr@u@xEIbABh@CXFn@?d@RxBB|@H`AAj@@n@Ef@EvAG~@FfBC~@KdA?\JzB?t@CZFv@Cv@J|A@p@JfCNdBGn@Ot@CbAQnAKpB@xAMp@Qd@Op@MfAi@`BIh@?\IjAMp@E\GTKr@Up@If@SxBAd@Db@AV@dA?JKVER?tCCb@I`@Ut@w@fBQh@KRQd@{@fBc@jAa@x@QRgArBa@fAu@~Ce@fESnCEbAUpB?f@Qr@Kr@]`AGr@ITIv@?j@IzABp@CdAOx@Cd@e@`CYhAc@|BIv@UhAkAdH"",3}",41326,4626,169,"{603,""2024-09-08"",""everyone"",12358663311,""everyone"",4}","{""5:46"",""7:18"",""5:46"",{""strava://segments/7116683/leaderboard"",""overall"",""All-Time""}}","{6149474,""Robert Šmíd"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/6149474/13314419/5/large.jpg"",""47 efforts in the last 90 days"",""47"",{""47 efforts"",""6 efforts""},""strava://segments/7116683/local_legend?categories%5B%5D=overall""}",2024-12-24 23:05:01 CET
8716919,3,"""tenis 1K""","""Run""",1002.8,-0.3,1.4,210.6,205.8,"[49.20352, 16.56572]","[49.210251, 16.557624]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/JMJ4PFQIMKN66VW23C3SPZ6EZKEEOJCKPAWPH4SB3ZTCAFOMBPEG4VJV4LEPQJNRVIDA7732HY4CCHC4LWFKESEB"",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/UPJFHIDOXYZWBQGX4JGB22F7WACQA4NILG5YN67BVT25VFRZ4QKISGUPCG7YSKOQXZOM65LE7AJNSBAXZAA4RQ4S""}",0,"""Brno""","""South Moravian Region""","""Czech Republic""",false,false,false,"""2015-01-03T12:00:16Z""","""2021-05-20T08:03:15Z""",0.0,"{""s8716919"",""_aikHwnbdBqBzCiAlAk@t@s@f@i@z@k@r@w@^y@NoA`@WTy@d@WVYN[Bw@`@Q`@]nAQf@}@|AIh@MvAQtAu@j@]Hy@Jw@j@m@t@k@z@S^_@lAWtA_@jA"",3}",48666,2441,34,"{220,""2022-11-30"",""everyone"",8189229964,""everyone"",53}","{""2:53"",""3:28"",""2:53"",{""strava://segments/8716919/leaderboard"",""overall"",""All-Time""}}","{31296919,""Jan Pavelka"",""https://dgalywyr863hv.cloudfront.net/pictures